# RankMixer 排序对比实验

用 PyTorch 版 RankMixer 替代 JRC 中的 MLP backbone + DIN，跳过手动特征交叉，
直接从 recall 结果 + 原始数据构建轻量数据管道。

**与 JRC 的关键差异：**

| 维度 | JRC | RankMixer |
|------|-----|----------|
| 特征工程 | 20 个手动交叉特征 | 4 个原子标量 + 64 维 content emb |
| 历史序列 | DIN attention (8维 ID emb) | 位置编码 + mean pool (64维 content emb) |
| Backbone | MLP (256→128) | RankMixer (4 token × 64 dim × 2 layers) |
| 损失 | gate + BPR/CE | **balanced_ge_loss**：按 user 聚合 CE+GE（与 jrc ge_loss 实验区一致） |
| 输入总维度 | 100 | ~204 |

In [1]:
import copy
import gc
import logging
import pickle
import random
import time
import warnings
from collections import defaultdict
from pathlib import Path

warnings.filterwarnings('ignore')
logger = logging.getLogger(__name__)

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, ndcg_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 2020
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODE = "offline"
offline = (MODE == "offline")

print("=" * 70)
print(f"RankMixer 排序对比实验 - 模式: {MODE}")
print(f"设备: {DEVICE}")
print("=" * 70)

RankMixer 排序对比实验 - 模式: offline
设备: cpu


## 1. 数据加载

In [2]:
project_root = Path.cwd().resolve()
data_path = project_root / 'data' / 'raw' / 'news_recommendation'
save_path = project_root / 'data' / 'processed' / 'temp_results'
output_path = project_root / 'outputs'

for path in [data_path, save_path, output_path]:
    path.mkdir(parents=True, exist_ok=True)

print(f"项目根目录: {project_root}")
print(f"原始数据路径: {data_path}")
print(f"中间结果路径: {save_path}")

项目根目录: /Users/lixiang/Desktop/funrec-new-rec
原始数据路径: /Users/lixiang/Desktop/funrec-new-rec/data/raw/news_recommendation
中间结果路径: /Users/lixiang/Desktop/funrec-new-rec/data/processed/temp_results


In [3]:
# 读取召回结果
recall_path = save_path / ('offline_all_final_recall_items_dict.pkl' if offline else 'online_all_final_recall_items_dict.pkl')
print(f"读取多路召回列表: {recall_path}")
recall_list_dict = pickle.load(open(recall_path, 'rb'))
print(f"召回用户数: {len(recall_list_dict)}")

读取多路召回列表: /Users/lixiang/Desktop/funrec-new-rec/data/processed/temp_results/offline_all_final_recall_items_dict.pkl
召回用户数: 200000


In [4]:
# 读取历史点击、标签、用户划分
click_hist_all = pd.read_csv(save_path / 'click_hist_all.csv')
click_last_all = pd.read_csv(save_path / 'click_last_all.csv')
rank_user_split = pickle.load(open(save_path / 'rank_user_split_all.pkl', 'rb'))
click_tst = pd.read_csv(data_path / 'testA_click_log.csv')

train_users = set(rank_user_split['train_users'])
val_users = set(rank_user_split['val_users'])

click_trn_hist = click_hist_all[click_hist_all['user_id'].isin(train_users)].reset_index(drop=True)
click_val_hist = click_hist_all[click_hist_all['user_id'].isin(val_users)].reset_index(drop=True)
click_trn_last = click_last_all[click_last_all['user_id'].isin(train_users)].reset_index(drop=True)
click_val_last = click_last_all[click_last_all['user_id'].isin(val_users)].reset_index(drop=True)
click_tst_hist = click_tst

print(f"训练用户历史: {click_trn_hist.shape}, target: {click_trn_last.shape}")
print(f"验证用户历史: {click_val_hist.shape}, target: {click_val_last.shape}")
print(f"测试集历史: {click_tst_hist.shape}")

训练用户历史: (731822, 9), target: (160000, 9)
验证用户历史: (180801, 9), target: (40000, 9)
测试集历史: (518010, 9)


In [5]:
# 读取文章元数据和 content embedding
articles = pd.read_csv(data_path / 'articles.csv')
print(f"文章元数据: {articles.shape}")

item_content_emb_dict = pickle.load(open(save_path / 'item_content_emb_all.pkl', 'rb'))
sample_key = next(iter(item_content_emb_dict))
raw_emb_dim = len(item_content_emb_dict[sample_key])
print(f"Content embedding: {len(item_content_emb_dict)} 篇文章, 原始维度={raw_emb_dim}")

文章元数据: (364047, 4)
Content embedding: 364047 篇文章, 原始维度=250


In [6]:
# PCA 降维 content embedding: 250 -> 64
CONTENT_EMB_DIM = 64

all_article_ids = sorted(item_content_emb_dict.keys())
raw_emb_matrix = np.stack([item_content_emb_dict[aid] for aid in all_article_ids])

pca = PCA(n_components=CONTENT_EMB_DIM, random_state=SEED)
reduced_emb_matrix = pca.fit_transform(raw_emb_matrix)
print(f"PCA 降维: {raw_emb_matrix.shape} -> {reduced_emb_matrix.shape}")
print(f"保留方差比例: {pca.explained_variance_ratio_.sum():.4f}")

# 重建字典: article_id -> 64维向量
article_emb_dict = {aid: reduced_emb_matrix[i].astype(np.float32) for i, aid in enumerate(all_article_ids)}

# 构建连续索引映射: article_id -> idx (0 保留给 padding)
article_id_to_idx = {aid: i + 1 for i, aid in enumerate(all_article_ids)}
num_articles = len(all_article_ids) + 1  # +1 for padding

# 构建 embedding 矩阵 (用于查表)
emb_table = np.zeros((num_articles, CONTENT_EMB_DIM), dtype=np.float32)
for aid, idx in article_id_to_idx.items():
    emb_table[idx] = article_emb_dict[aid]

print(f"Embedding 查表矩阵: {emb_table.shape}")

del raw_emb_matrix, reduced_emb_matrix, item_content_emb_dict
gc.collect()

PCA 降维: (364047, 250) -> (364047, 64)
保留方差比例: 0.9717
Embedding 查表矩阵: (364048, 64)


0

## 2. 轻量数据管道（替代 feature_engineering）

In [7]:
def recall_dict_2_df(recall_list_dict):
    df_row_list = []
    for user, recall_list in recall_list_dict.items():
        if isinstance(recall_list, dict):
            recall_list = recall_list.items()
        for item, score in recall_list:
            df_row_list.append([user, item, score])
    return pd.DataFrame(df_row_list, columns=['user_id', 'sim_item', 'score'])


def attach_recall_rank(recall_df):
    recall_df = recall_df.copy()
    recall_df['recall_rank'] = (
        recall_df.groupby('user_id')['score']
        .rank(ascending=False, method='first')
        .astype(int) - 1
    )
    return recall_df


def get_rank_label_df(recall_list_df, label_df, is_test=False):
    if is_test:
        recall_list_df['label'] = -1
        return recall_list_df
    label_df = label_df.rename(columns={'click_article_id': 'sim_item'})
    recall_list_df_ = recall_list_df.merge(
        label_df[['user_id', 'sim_item', 'click_timestamp']],
        how='left', on=['user_id', 'sim_item']
    )
    recall_list_df_['label'] = recall_list_df_['click_timestamp'].apply(
        lambda x: 0.0 if np.isnan(x) else 1.0
    )
    del recall_list_df_['click_timestamp']
    return recall_list_df_


def neg_sample_recall_data(
    recall_items_df,
    top_neg_num=2, mid_neg_num=3, deep_neg_num=3, tail_neg_num=4,
    seed=2024,
):
    print("负采样前:")
    print(recall_items_df['label'].value_counts(dropna=False))

    def sample_bucket(bucket_df, sample_num, random_state):
        if sample_num <= 0 or len(bucket_df) == 0:
            return bucket_df.iloc[0:0]
        if len(bucket_df) <= sample_num:
            return bucket_df
        return bucket_df.sample(n=sample_num, replace=False, random_state=random_state)

    def user_neg_sample_func(user_df):
        pos_df = user_df[user_df['label'] == 1]
        neg_df = user_df[user_df['label'] == 0].copy()
        if len(pos_df) == 0:
            return user_df.iloc[0:0]
        if 'recall_rank' not in neg_df.columns:
            neg_df = neg_df.sort_values('score', ascending=False).copy()
            neg_df['recall_rank'] = np.arange(len(neg_df))
        user_seed = seed + int(user_df['user_id'].iloc[0])
        sampled_neg_df = pd.concat([
            sample_bucket(neg_df[neg_df['recall_rank'].between(0, 9)], top_neg_num, user_seed + 1),
            sample_bucket(neg_df[neg_df['recall_rank'].between(10, 49)], mid_neg_num, user_seed + 2),
            sample_bucket(neg_df[neg_df['recall_rank'].between(50, 99)], deep_neg_num, user_seed + 3),
            sample_bucket(neg_df[neg_df['recall_rank'] >= 100], tail_neg_num, user_seed + 4),
        ], ignore_index=False)
        sampled_neg_df = sampled_neg_df.sort_values(['recall_rank', 'score'], ascending=[True, False])
        sampled_neg_df = sampled_neg_df.drop_duplicates(['user_id', 'sim_item'])
        return pd.concat([pos_df, sampled_neg_df], ignore_index=True)

    data_new = (
        recall_items_df.groupby('user_id', group_keys=False)
        .apply(user_neg_sample_func)
        .reset_index(drop=True)
    )
    print("负采样后:")
    print(data_new['label'].value_counts(dropna=False))
    return data_new


print("数据管道函数定义完成")

数据管道函数定义完成


In [8]:
# 召回列表转 DataFrame + 打标签 + 负采样
print("转换召回列表...")
recall_list_df = recall_dict_2_df(recall_list_dict)
print(f"召回列表总条数: {len(recall_list_df)}")

# --- 训练集 ---
trn_users = click_trn_hist['user_id'].unique()
trn_recall_df = recall_list_df[recall_list_df['user_id'].isin(trn_users)].copy()
trn_recall_df = attach_recall_rank(trn_recall_df)
trn_label_df = get_rank_label_df(trn_recall_df, click_trn_last, is_test=False)

user_has_pos = trn_label_df.groupby('user_id')['label'].transform('max')
trn_label_df_hit = trn_label_df[user_has_pos == 1].copy()
print(f"训练集命中用户数: {trn_label_df_hit['user_id'].nunique()}, 总样本: {len(trn_label_df_hit)}")

print("\n训练集负采样:")
trn_df = neg_sample_recall_data(trn_label_df_hit)
trn_df = trn_df.rename(columns={'sim_item': 'click_article_id'})
print(f"训练集最终: {trn_df.shape}")

# --- 验证集 ---
if offline:
    val_recall_users = click_val_hist['user_id'].unique()
    val_recall_df = recall_list_df[recall_list_df['user_id'].isin(val_recall_users)].copy()
    val_recall_df = attach_recall_rank(val_recall_df)
    val_df = get_rank_label_df(val_recall_df, click_val_last, is_test=False)
    val_df = val_df.rename(columns={'sim_item': 'click_article_id'})
    print(f"验证集: {val_df.shape}")
else:
    val_df = None

# --- 测试集 ---
tst_users = click_tst_hist['user_id'].unique()
tst_recall_df = recall_list_df[recall_list_df['user_id'].isin(tst_users)].copy()
tst_recall_df = attach_recall_rank(tst_recall_df)
tst_df = get_rank_label_df(tst_recall_df, None, is_test=True)
tst_df = tst_df.rename(columns={'sim_item': 'click_article_id'})
print(f"测试集: {tst_df.shape}")

del recall_list_dict, recall_list_df
gc.collect()

转换召回列表...
召回列表总条数: 40000000
训练集命中用户数: 109760, 总样本: 21952000

训练集负采样:
负采样前:
label
0.0    21842240
1.0      109760
Name: count, dtype: int64
负采样后:
label
0.0    1317120
1.0     109760
Name: count, dtype: int64
训练集最终: (1426880, 5)
验证集: (8000000, 5)
测试集: (0, 5)


0

In [9]:
# 拼接文章元数据 (category_id, created_at_ts, words_count)
article_meta = articles[['article_id', 'category_id', 'created_at_ts', 'words_count']].copy()

trn_df = trn_df.merge(article_meta, left_on='click_article_id', right_on='article_id', how='left')
if 'article_id' in trn_df.columns:
    del trn_df['article_id']

if offline and val_df is not None:
    val_df = val_df.merge(article_meta, left_on='click_article_id', right_on='article_id', how='left')
    if 'article_id' in val_df.columns:
        del val_df['article_id']

if len(tst_df) > 0:
    tst_df = tst_df.merge(article_meta, left_on='click_article_id', right_on='article_id', how='left')
    if 'article_id' in tst_df.columns:
        del tst_df['article_id']

# 构建用户历史序列 (user_id -> [article_id_list])
hist_click = click_hist_all[['user_id', 'click_article_id']].groupby('user_id').agg({list}).reset_index()
his_behavior_df = pd.DataFrame()
his_behavior_df['user_id'] = hist_click['user_id']
his_behavior_df['hist_click_article_id'] = hist_click['click_article_id']

# 拼接用户设备信息 (取众数)
device_cols = ['user_id', 'click_environment', 'click_deviceGroup', 'click_os',
               'click_country', 'click_region', 'click_referrer_type']
user_device = click_hist_all[device_cols].groupby('user_id').agg(
    lambda x: x.value_counts().index[0]
).reset_index()

# 合并到特征表
trn_df = trn_df.merge(his_behavior_df, on='user_id', how='left')
trn_df = trn_df.merge(user_device, on='user_id', how='left')

if offline and val_df is not None:
    val_df = val_df.merge(his_behavior_df, on='user_id', how='left')
    val_df = val_df.merge(user_device, on='user_id', how='left')

if len(tst_df) > 0:
    tst_df = tst_df.merge(his_behavior_df, on='user_id', how='left')
    tst_df = tst_df.merge(user_device, on='user_id', how='left')

print(f"训练集列: {trn_df.columns.tolist()}")
print(f"训练集形状: {trn_df.shape}")
if val_df is not None:
    print(f"验证集形状: {val_df.shape}")

训练集列: ['user_id', 'click_article_id', 'score', 'recall_rank', 'label', 'category_id', 'created_at_ts', 'words_count', 'hist_click_article_id', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type']
训练集形状: (1426880, 15)
验证集形状: (8000000, 15)


In [10]:
# val_hit_mask 用于 hit-only 评估
if offline and val_df is not None:
    val_hit_mask = (val_df.groupby('user_id')['label'].transform('max') == 1)
    val_df_hit = val_df[val_hit_mask].copy()
    print(f"验证集 hit 用户数: {val_df_hit['user_id'].nunique()}")
    print(f"验证集 hit 样本数: {len(val_df_hit)}")
else:
    val_hit_mask = None
    val_df_hit = None

验证集 hit 用户数: 27382
验证集 hit 样本数: 5476400


## 3. 特征处理与模型输入构建

In [11]:
# 特征定义
sparse_fea = [
    'user_id', 'click_article_id', 'category_id',
    'click_environment', 'click_deviceGroup', 'click_os',
    'click_country', 'click_region', 'click_referrer_type',
]

# 只保留原子标量特征，不做手动交叉
dense_fea = ['score', 'recall_rank', 'created_at_ts', 'words_count']

EMB_DIM = 8
MAX_HIST_LEN = 50

# Dense 特征归一化
mm_scalers = {}
for feat in dense_fea:
    mm = MinMaxScaler()
    trn_df[feat] = trn_df[feat].astype('float32').fillna(0)
    trn_df[feat] = mm.fit_transform(trn_df[[feat]])
    mm_scalers[feat] = mm
    if offline and val_df is not None:
        val_df[feat] = val_df[feat].astype('float32').fillna(0)
        val_df[feat] = mm.transform(val_df[[feat]])
    if len(tst_df) > 0:
        tst_df[feat] = tst_df[feat].astype('float32').fillna(0)
        tst_df[feat] = mm.transform(tst_df[[feat]])

print(f"稀疏特征 ({len(sparse_fea)}): {sparse_fea}")
print(f"稠密特征 ({len(dense_fea)}): {dense_fea}")
print("Dense 特征归一化完成")

稀疏特征 (9): ['user_id', 'click_article_id', 'category_id', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type']
稠密特征 (4): ['score', 'recall_rank', 'created_at_ts', 'words_count']
Dense 特征归一化完成


In [12]:
# 稀疏特征 ID 映射
all_frames = [trn_df] + ([val_df] if val_df is not None else []) + ([tst_df] if len(tst_df) > 0 else [])

sparse_maps = {}
for feat in sparse_fea:
    vals = pd.concat([f[feat].astype('int64') for f in all_frames], ignore_index=True)
    if feat == 'click_article_id':
        hist_vals = []
        for f in all_frames:
            for seq in f['hist_click_article_id']:
                if isinstance(seq, (list, np.ndarray)):
                    hist_vals.extend([int(x) for x in seq])
        if len(hist_vals) > 0:
            vals = pd.concat([vals, pd.Series(hist_vals, dtype='int64')], ignore_index=True)
    _, uniques = pd.factorize(vals, sort=False)
    mapping = {int(val): int(idx) for idx, val in enumerate(uniques)}
    if feat == 'click_article_id':
        mapping = {k: v + 1 for k, v in mapping.items()}
    sparse_maps[feat] = mapping

sparse_vocab_sizes = {
    feat: max(sparse_maps[feat].values()) + 1 for feat in sparse_fea
}
dense_input_dim = len(dense_fea)

print("稀疏特征 ID 映射构建完成")
for feat in sparse_fea:
    print(f"  {feat}: vocab_size = {sparse_vocab_sizes[feat]}")

稀疏特征 ID 映射构建完成
  user_id: vocab_size = 149760
  click_article_id: vocab_size = 22783
  category_id: vocab_size = 248
  click_environment: vocab_size = 3
  click_deviceGroup: vocab_size = 4
  click_os: vocab_size = 8
  click_country: vocab_size = 11
  click_region: vocab_size = 28
  click_referrer_type: vocab_size = 7


In [13]:
def pad_sequences_np(sequences, maxlen, value=0):
    padded = np.full((len(sequences), maxlen), value, dtype=np.int64)
    for idx, seq in enumerate(sequences):
        if len(seq) == 0:
            continue
        trunc = seq[:maxlen]
        padded[idx, :len(trunc)] = np.asarray(trunc, dtype=np.int64)
    return padded


def build_model_input(df):
    """构建 RankMixer 模型输入字典"""
    model_input = {}

    # 稀疏特征 -> 映射后 ID
    for feat in sparse_fea:
        model_input[feat] = (
            df[feat].astype('int64').map(sparse_maps[feat]).fillna(0).astype('int64').values
        )

    # 稠密特征
    for feat in dense_fea:
        model_input[feat] = df[feat].astype('float32').values

    # 候选文章 content embedding (64维)
    candidate_ids = df['click_article_id'].astype(int).values
    candidate_embs = np.stack([
        article_emb_dict.get(aid, np.zeros(CONTENT_EMB_DIM, dtype=np.float32))
        for aid in candidate_ids
    ])
    model_input['candidate_emb'] = candidate_embs

    # 历史序列: 转为 article_id_to_idx 索引 (用于查 emb_table)
    raw_seq_list = df['hist_click_article_id'].tolist()
    mapped_seq = []
    for seq in raw_seq_list:
        if isinstance(seq, (list, np.ndarray)):
            mapped_seq.append([article_id_to_idx.get(int(x), 0) for x in seq])
        else:
            mapped_seq.append([0])
    model_input['hist_article_idx'] = pad_sequences_np(mapped_seq, MAX_HIST_LEN, value=0)

    return model_input


print("构建模型输入...")
x_trn = build_model_input(trn_df)
y_trn = trn_df['label'].values.astype('float32')
print(f"训练集: {len(y_trn)} 样本, 正样本率: {y_trn.mean():.4f}")

if offline and val_df is not None:
    x_val = build_model_input(val_df)
    y_val = val_df['label'].values.astype('float32')
    print(f"验证集: {len(y_val)} 样本, 正样本率: {y_val.mean():.4f}")
else:
    x_val, y_val = None, None

print("模型输入构建完成")

构建模型输入...
训练集: 1426880 样本, 正样本率: 0.0769
验证集: 8000000 样本, 正样本率: 0.0034
模型输入构建完成


## 4. PyTorch RankMixer 模型

In [14]:
# ====== RankMixer 核心模块 (PyTorch) ======

class SemanticTokenization(nn.Module):
    """将 flat 向量拆成 num_T 段，每段映射到 num_D 维"""
    def __init__(self, input_dim, num_T, num_D):
        super().__init__()
        self.num_T = num_T
        chunk_dim = input_dim // num_T
        self.remainder = input_dim - chunk_dim * num_T
        self.linears = nn.ModuleList([
            nn.Linear(chunk_dim + (1 if i < self.remainder else 0), num_D)
            for i in range(num_T)
        ])

    def forward(self, x):
        if self.remainder == 0:
            chunks = torch.chunk(x, self.num_T, dim=-1)
        else:
            chunk_dim = x.size(-1) // self.num_T
            chunks = []
            start = 0
            for i in range(self.num_T):
                size = chunk_dim + (1 if i < self.remainder else 0)
                chunks.append(x[:, start:start + size])
                start += size
        tokens = [self.linears[i](chunks[i]) for i in range(self.num_T)]
        return torch.stack(tokens, dim=1)  # (B, T, D)


class TokenMixer(nn.Module):
    """Multi-head reshape mixer (无可学习参数)"""
    def __init__(self, num_T, num_D, num_H):
        super().__init__()
        self.num_T = num_T
        self.num_H = num_H
        self.d_k = num_D // num_H

    def forward(self, x):
        B = x.size(0)
        x = x.view(B, self.num_T, self.num_H, self.d_k)
        x = x.permute(0, 2, 1, 3)
        return x.reshape(B, self.num_H, self.num_T * self.d_k)


class PerTokenFFN(nn.Module):
    """每个 token 独立的 FFN (MoE 风格)"""
    def __init__(self, num_T, num_D, expansion_ratio=4):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(num_D, num_D * expansion_ratio),
                nn.GELU(),
                nn.Linear(num_D * expansion_ratio, num_D),
            )
            for _ in range(num_T)
        ])

    def forward(self, x):
        return torch.stack(
            [self.experts[i](x[:, i]) for i in range(len(self.experts))],
            dim=1,
        )


class RankMixerLayer(nn.Module):
    def __init__(self, num_T, num_D, num_H, expansion_ratio=4):
        super().__init__()
        self.token_mixer = TokenMixer(num_T, num_D, num_H)
        self.per_token_ffn = PerTokenFFN(num_T, num_D, expansion_ratio)
        self.norm1 = nn.LayerNorm(num_D)
        self.norm2 = nn.LayerNorm(num_D)

    def forward(self, x):
        mixed = self.token_mixer(x)
        x = self.norm1(x + mixed)
        x = self.norm2(x + self.per_token_ffn(x))
        return x


class RankMixerCore(nn.Module):
    """RankMixer: SemanticTokenization + N x RankMixerLayer + mean pool"""
    def __init__(self, input_dim, num_layers, num_T, num_D, num_H, expansion_ratio=4):
        super().__init__()
        self.tokenization = SemanticTokenization(input_dim, num_T, num_D)
        self.layers = nn.ModuleList([
            RankMixerLayer(num_T, num_D, num_H, expansion_ratio)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        x = self.tokenization(x)  # (B, T, D)
        for layer in self.layers:
            x = layer(x)
        return x.mean(dim=1)  # (B, D)


print("RankMixer 核心模块定义完成")

RankMixer 核心模块定义完成


In [15]:
# ====== RankMixer Ranking 主模型 ======

def make_mlp(input_dim, hidden_units, dropout_rate=0.0, output_dim=None):
    layers = []
    prev = input_dim
    for units in hidden_units:
        layers += [nn.Linear(prev, units), nn.ReLU()]
        if dropout_rate > 0:
            layers.append(nn.Dropout(dropout_rate))
        prev = units
    if output_dim is not None:
        layers.append(nn.Linear(prev, output_dim))
        prev = output_dim
    return nn.Sequential(*layers), prev


class RankMixerRanking(nn.Module):
    def __init__(self, sparse_vocab_sizes, dense_input_dim, config):
        super().__init__()
        self.sparse_fea_names = list(sparse_vocab_sizes.keys())
        self.dense_fea_names = dense_fea
        emb_dim = config['emb_dim']
        content_emb_dim = config['content_emb_dim']
        max_hist_len = config['max_hist_len']
        self.use_linear = config.get('use_linear', True)

        # Sparse embeddings
        self.sparse_embeddings = nn.ModuleDict()
        for feat, vocab_size in sparse_vocab_sizes.items():
            padding_idx = 0 if feat == 'click_article_id' else None
            self.sparse_embeddings[feat] = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)

        # Linear logits 分支
        if self.use_linear:
            self.linear_sparse_embeddings = nn.ModuleDict({
                feat: nn.Embedding(vs, 1, padding_idx=0 if feat == 'click_article_id' else None)
                for feat, vs in sparse_vocab_sizes.items()
            })
            self.linear_dense = nn.Linear(dense_input_dim + content_emb_dim, 1)

        # Content embedding 查表 (frozen)
        self.content_emb_table = nn.Embedding.from_pretrained(
            torch.tensor(emb_table, dtype=torch.float32), freeze=True, padding_idx=0
        )

        # 可学习位置编码
        self.position_emb = nn.Embedding(max_hist_len, content_emb_dim)

        # RankMixer backbone
        sparse_total = len(self.sparse_fea_names) * emb_dim
        hist_repr_dim = content_emb_dim
        rankmixer_input_dim = sparse_total + dense_input_dim + content_emb_dim + hist_repr_dim
        self.rankmixer_input_dim = rankmixer_input_dim

        self.rankmixer = RankMixerCore(
            input_dim=rankmixer_input_dim,
            num_layers=config['num_layers'],
            num_T=config['num_T'],
            num_D=config['num_D'],
            num_H=config['num_H'],
            expansion_ratio=config['expansion_ratio'],
        )

        # Logit head -> 2D output
        self.logit_head, logit_out_dim = make_mlp(
            config['num_D'],
            config.get('logit_head_units', [32]),
            dropout_rate=config.get('dropout_rate', 0.0),
        )
        self.logits_out = nn.Linear(logit_out_dim, 2)

    def forward(self, batch):
        # Sparse embeddings
        sparse_emb_list = [
            self.sparse_embeddings[feat](batch[feat])
            for feat in self.sparse_fea_names
        ]

        # Dense features
        dense_tensor = torch.stack([batch[feat] for feat in self.dense_fea_names], dim=1)

        # Candidate content embedding
        candidate_emb = batch['candidate_emb']  # (B, 64)

        # Historical sequence with positional encoding
        hist_idx = batch['hist_article_idx']  # (B, L)
        hist_mask = hist_idx.ne(0)  # (B, L)
        hist_emb = self.content_emb_table(hist_idx)  # (B, L, 64)

        B, L, D = hist_emb.shape
        positions = torch.arange(L, device=hist_emb.device).unsqueeze(0).expand(B, -1)
        hist_emb = hist_emb + self.position_emb(positions)

        # Masked mean pooling
        hist_mask_f = hist_mask.float().unsqueeze(-1)  # (B, L, 1)
        hist_repr = (hist_emb * hist_mask_f).sum(dim=1) / hist_mask_f.sum(dim=1).clamp_min(1.0)

        # Concat all -> RankMixer
        dnn_input = torch.cat(sparse_emb_list + [dense_tensor, candidate_emb, hist_repr], dim=1)
        rankmixer_out = self.rankmixer(dnn_input)

        # Logit head
        logits_hidden = self.logit_head(rankmixer_out)
        logits_2d = self.logits_out(logits_hidden)

        # Linear logits 分支
        if self.use_linear:
            linear_sparse = sum(
                self.linear_sparse_embeddings[feat](batch[feat]).squeeze(-1)
                for feat in self.sparse_fea_names
            )
            linear_dense_input = torch.cat([dense_tensor, candidate_emb], dim=1)
            linear_dense = self.linear_dense(linear_dense_input).squeeze(-1)
            linear_logit = linear_sparse + linear_dense
            linear_2d = torch.stack([-linear_logit, linear_logit], dim=1)
            logits_2d = logits_2d + linear_2d

        return logits_2d


print("RankMixerRanking 模型定义完成")

RankMixerRanking 模型定义完成


## 5. 损失函数、训练与评估

In [16]:
# ====== 损失函数（与 jrc-ranking_all.ipynb ge_loss 实验区一致）======
# paper_context_ge_loss_balanced + 按 user 聚合：
#   total_loss = sum(total_terms) / denom
#   ce_loss = sum(ce_terms) / denom
#   ge_loss = sum(ge_terms) / denom
# 等价于 compute_joint_loss_ge_balanced_fulluser(..., is_hit_context=None)。

def paper_context_ge_loss_balanced(
    logits_2d,
    labels_onehot,
    context_ids,
    pos_weight=1.0,
    neg_weight=1.0,
):
    batch_size = logits_2d.size(0)
    if batch_size == 0:
        return logits_2d.new_tensor(0.0)

    mask = context_ids.unsqueeze(0).eq(context_ids.unsqueeze(1))
    logits_tile = logits_2d.unsqueeze(1).expand(-1, batch_size, -1).clone()
    labels_tile = labels_onehot.unsqueeze(1).expand(-1, batch_size, -1).float()
    invalid_mask = ~mask.unsqueeze(-1)
    logits_tile = logits_tile.masked_fill(invalid_mask, -1e9)
    labels_tile = labels_tile * mask.unsqueeze(-1).float()

    l_neg = logits_tile[:, :, 0]
    l_pos = logits_tile[:, :, 1]
    y_neg = labels_tile[:, :, 0]
    y_pos = labels_tile[:, :, 1]

    log_softmax_pos = F.log_softmax(l_pos, dim=0)
    log_softmax_neg = F.log_softmax(l_neg, dim=0)

    loss_pos = -(y_pos * log_softmax_pos).sum(dim=0)
    loss_neg = -(y_neg * log_softmax_neg).sum(dim=0)

    pos_cnt = y_pos.sum(dim=0).clamp_min(1.0)
    neg_cnt = y_neg.sum(dim=0).clamp_min(1.0)

    ge_loss = (
        pos_weight * loss_pos / pos_cnt +
        neg_weight * loss_neg / neg_cnt
    ).mean()
    return ge_loss


def compute_rankmixer_joint_loss(
    logits_2d,
    y_true,
    context_ids,
    alpha=0.5,
    pos_weight=1.0,
    neg_weight=1.0,
):
    """与 jrc `compute_joint_loss_ge_balanced_fulluser` 在 is_hit_context=None 时相同。"""
    total_terms = []
    ce_terms = []
    ge_terms = []
    weight_terms = []

    for uid in torch.unique(context_ids):
        mask = context_ids == uid
        logits_ctx = logits_2d[mask]
        y_ctx = y_true[mask]
        local_context_ids = torch.zeros_like(y_ctx, dtype=context_ids.dtype)

        ce_ctx = F.cross_entropy(logits_ctx, y_ctx.long())
        y_ctx_long = y_ctx.long()
        y_ctx_neg = 1 - y_ctx_long
        labels_onehot_ctx = torch.stack([y_ctx_neg, y_ctx_long], dim=1).float()
        ge_ctx = paper_context_ge_loss_balanced(
            logits_ctx,
            labels_onehot_ctx,
            local_context_ids,
            pos_weight=pos_weight,
            neg_weight=neg_weight,
        )
        total_ctx = alpha * ce_ctx + (1.0 - alpha) * ge_ctx
        ctx_weight = 1.0

        total_terms.append(total_ctx * ctx_weight)
        ce_terms.append(ce_ctx * ctx_weight)
        ge_terms.append(ge_ctx * ctx_weight)
        weight_terms.append(ctx_weight)

    if not weight_terms:
        zero = logits_2d.new_tensor(0.0)
        return zero, zero, zero

    denom = logits_2d.new_tensor(sum(weight_terms)).clamp_min(1e-8)
    total_loss = torch.stack(total_terms).sum() / denom
    ce_loss = torch.stack(ce_terms).sum() / denom
    ge_loss = torch.stack(ge_terms).sum() / denom
    return total_loss, ce_loss, ge_loss


print("损失函数定义完成（balanced_ge_loss + 按 user 聚合，与 jrc ge_loss 实验区一致）")

损失函数定义完成（balanced_ge_loss + 按 user 聚合，与 jrc ge_loss 实验区一致）


In [17]:
# ====== Dataset / DataLoader ======

class RankingDictDataset(Dataset):
    def __init__(self, x_dict, y=None):
        self.x_dict = x_dict
        self.y = y
        self.length = len(next(iter(x_dict.values())))

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        sample = {key: value[idx] for key, value in self.x_dict.items()}
        if self.y is None:
            return sample
        return sample, self.y[idx]


class UserGroupedBatchSampler:
    def __init__(self, user_ids, batch_size=256, shuffle=True):
        self.batch_size = batch_size
        self.shuffle = shuffle
        user_to_indices = defaultdict(list)
        for idx, uid in enumerate(np.asarray(user_ids)):
            user_to_indices[int(uid)].append(idx)
        self.user_blocks = list(user_to_indices.values())

    def __iter__(self):
        blocks = list(self.user_blocks)
        if self.shuffle:
            random.shuffle(blocks)
        batch = []
        for block in blocks:
            if len(block) >= self.batch_size:
                if batch:
                    yield batch
                    batch = []
                yield block
                continue
            if len(batch) + len(block) > self.batch_size:
                if batch:
                    yield batch
                batch = list(block)
            else:
                batch.extend(block)
        if batch:
            yield batch

    def __len__(self):
        count, batch = 0, []
        for block in self.user_blocks:
            if len(block) >= self.batch_size:
                if batch:
                    count += 1
                    batch = []
                count += 1
                continue
            if len(batch) + len(block) > self.batch_size:
                count += 1
                batch = list(block)
            else:
                batch.extend(block)
        if batch:
            count += 1
        return count


def collate_with_labels(batch):
    features, labels = zip(*batch)
    collated = {}
    for key in features[0]:
        arr = np.stack([item[key] for item in features])
        if key in sparse_fea or key == 'hist_article_idx':
            collated[key] = torch.tensor(arr, dtype=torch.long)
        else:
            collated[key] = torch.tensor(arr, dtype=torch.float32)
    return collated, torch.tensor(np.asarray(labels), dtype=torch.float32)


def collate_features(batch):
    collated = {}
    for key in batch[0]:
        arr = np.stack([item[key] for item in batch])
        if key in sparse_fea or key == 'hist_article_idx':
            collated[key] = torch.tensor(arr, dtype=torch.long)
        else:
            collated[key] = torch.tensor(arr, dtype=torch.float32)
    return collated


def make_loader(x_dict, y=None, batch_size=256, shuffle=False, group_by_user=False):
    dataset = RankingDictDataset(x_dict, y)
    if y is None:
        return DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_features)
    if group_by_user:
        sampler = UserGroupedBatchSampler(x_dict['user_id'], batch_size=batch_size, shuffle=shuffle)
        return DataLoader(dataset, batch_sampler=sampler, collate_fn=collate_with_labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_with_labels)


def move_batch_to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}


print("DataLoader 工具定义完成")

DataLoader 工具定义完成


In [18]:
# ====== 评估函数 (与 JRC 完全一致) ======

def safe_auc(labels, preds):
    labels = np.asarray(labels)
    if np.unique(labels).size < 2:
        return float('nan')
    return float(roc_auc_score(labels, preds))


def compute_ndcg_at_k(labels, preds, user_id_list, k=5):
    group_score = defaultdict(list)
    group_truth = defaultdict(list)
    for idx, uid in enumerate(user_id_list):
        group_score[uid].append(preds[idx])
        group_truth[uid].append(labels[idx])
    ndcg_list = []
    for uid in group_score:
        y_true = np.array([group_truth[uid]])
        y_score = np.array([group_score[uid]])
        if np.sum(y_true) == 0:
            ndcg_list.append(0.0)
        else:
            ndcg_list.append(ndcg_score(y_true, y_score, k=k))
    return float(np.mean(ndcg_list))


def compute_mrr_hr_stats(df, topk=5, score_col='pred_score'):
    df_sorted = df.sort_values(by=['user_id', score_col], ascending=[True, False])
    mrr_sum = hr_sum = user_count = pos_user_count = 0
    for _, group in df_sorted.groupby('user_id'):
        user_count += 1
        labels = group['label'].values
        if labels.sum() == 0:
            continue
        pos_user_count += 1
        pos_idx = np.where(labels == 1)[0]
        if len(pos_idx) > 0:
            rank = pos_idx[0] + 1
            if rank <= topk:
                mrr_sum += 1.0 / rank
                hr_sum += 1
    mrr = mrr_sum / user_count if user_count > 0 else 0.0
    hr = hr_sum / user_count if user_count > 0 else 0.0
    return {'mrr5': float(mrr), 'hr5': float(hr), 'pos_user_count': int(pos_user_count), 'user_count': int(user_count)}


def softmax_second_col(logits_2d):
    shifted = logits_2d - logits_2d.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    return probs[:, 1]


def build_predictions(logits_2d, mode='calibration'):
    ranking_scores = logits_2d[:, 1] - logits_2d[:, 0]
    calib_probs = softmax_second_col(logits_2d)
    if mode == 'calibration':
        return calib_probs
    r_min, r_max = ranking_scores.min(), ranking_scores.max()
    if r_max > r_min:
        norm_rank = (ranking_scores - r_min) / (r_max - r_min)
    else:
        norm_rank = ranking_scores
    if mode == 'ranking':
        return norm_rank
    if mode == 'fusion':
        return 0.5 * norm_rank + 0.5 * calib_probs
    raise ValueError(f"Unknown mode: {mode}")


def evaluate_prediction_slice(labels, preds, eval_df, topk=5):
    eval_frame = eval_df[['user_id', 'click_article_id', 'label']].copy()
    eval_frame['pred_score'] = preds
    rank_metrics = compute_mrr_hr_stats(eval_frame, topk=topk)
    return {
        'auc': safe_auc(labels, preds),
        'ndcg5': compute_ndcg_at_k(labels, preds, eval_df['user_id'].values, k=topk),
        'mrr5': rank_metrics['mrr5'],
        'hr5': rank_metrics['hr5'],
        'pos_user_count': rank_metrics['pos_user_count'],
        'user_count': rank_metrics['user_count'],
    }


def raw_predict(model, x_data, batch_size=256):
    loader = make_loader(x_data, y=None, batch_size=batch_size, shuffle=False)
    model.eval()
    logits_list = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch)
            logits_2d = model(batch)
            logits_list.append(logits_2d.cpu().numpy())
    return np.concatenate(logits_list, axis=0)


def evaluate_validation(
    model, x_val, y_val, val_df, val_df_hit=None, val_hit_mask=None,
    batch_size=256, topk=5, mode='fusion',
):
    logits_2d = raw_predict(model, x_val, batch_size=batch_size)
    preds = build_predictions(logits_2d, mode=mode)
    metrics = {}

    full_metrics = evaluate_prediction_slice(y_val, preds, val_df, topk=topk)
    for k, v in full_metrics.items():
        metrics[f'full_{k}'] = v

    if val_df_hit is not None and len(val_df_hit) > 0 and val_hit_mask is not None:
        hit_mask_np = val_hit_mask.to_numpy() if hasattr(val_hit_mask, 'to_numpy') else np.asarray(val_hit_mask)
        hit_preds = preds[hit_mask_np]
        hit_labels = val_df_hit['label'].values
        hit_metrics = evaluate_prediction_slice(hit_labels, hit_preds, val_df_hit, topk=topk)
        for k, v in hit_metrics.items():
            metrics[f'hit_{k}'] = v
    else:
        for k in ['auc', 'ndcg5', 'mrr5', 'hr5']:
            metrics[f'hit_{k}'] = float('nan')

    return metrics


print("评估函数定义完成")

评估函数定义完成


In [19]:
# ====== 训练函数 ======

def train_rankmixer_model(
    model, x_train, y_train, x_val=None, y_val=None,
    val_df=None, val_df_hit=None, val_hit_mask=None,
    epochs=5, batch_size=256, lr=0.001,
    alpha=0.5, pos_weight=1.0, neg_weight=1.0, select_topk=5,
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = make_loader(
        x_train, y_train, batch_size=batch_size, shuffle=True, group_by_user=True,
    )

    history = {k: [] for k in [
        'total_loss', 'ce_loss', 'ge_loss',
        'full_auc', 'full_ndcg5', 'full_mrr5', 'full_hr5',
        'hit_auc', 'hit_ndcg5', 'hit_mrr5', 'hit_hr5',
    ]}
    best_metric = float('-inf')
    best_state_dict = None
    best_epoch = None

    for epoch in range(epochs):
        epoch_losses = {k: [] for k in ['total_loss', 'ce_loss', 'ge_loss']}
        t0 = time.time()
        model.train()

        for x_batch, y_batch in train_loader:
            x_batch = move_batch_to_device(x_batch)
            y_batch = y_batch.to(DEVICE)
            user_ids = x_batch['user_id']

            optimizer.zero_grad()
            logits_2d = model(x_batch)
            total_loss, ce_l, ge_l = compute_rankmixer_joint_loss(
                logits_2d, y_batch, user_ids,
                alpha=alpha, pos_weight=pos_weight, neg_weight=neg_weight,
            )
            total_loss.backward()
            optimizer.step()

            epoch_losses['total_loss'].append(total_loss.item())
            epoch_losses['ce_loss'].append(ce_l.item())
            epoch_losses['ge_loss'].append(ge_l.item())

        elapsed = time.time() - t0
        avg_losses = {k: np.mean(v) for k, v in epoch_losses.items()}
        for k, v in avg_losses.items():
            history[k].append(v)

        log_str = (
            f"Epoch {epoch + 1}/{epochs} ({elapsed:.1f}s)  "
            f"Total: {avg_losses['total_loss']:.4f}  "
            f"CE: {avg_losses['ce_loss']:.4f}  "
            f"GE: {avg_losses['ge_loss']:.4f}"
        )

        if x_val is not None and y_val is not None:
            eval_metrics = evaluate_validation(
                model, x_val, y_val, val_df,
                val_df_hit=val_df_hit, val_hit_mask=val_hit_mask,
                batch_size=batch_size, topk=select_topk, mode='fusion',
            )
            for k in ['full_auc', 'full_ndcg5', 'full_mrr5', 'full_hr5',
                       'hit_auc', 'hit_ndcg5', 'hit_mrr5', 'hit_hr5']:
                history[k].append(eval_metrics.get(k, float('nan')))

            log_str += (
                f"  | Full MRR@5: {eval_metrics.get('full_mrr5', 0):.5f}"
                f"  | Hit MRR@5: {eval_metrics.get('hit_mrr5', 0):.5f}"
                f"  | Full NDCG@5: {eval_metrics.get('full_ndcg5', 0):.4f}"
                f"  | Hit NDCG@5: {eval_metrics.get('hit_ndcg5', 0):.4f}"
            )

            select_val = eval_metrics.get('hit_mrr5', float('nan'))
            if np.isnan(select_val):
                select_val = eval_metrics.get('full_mrr5', float('-inf'))

            if select_val > best_metric:
                best_metric = select_val
                best_epoch = epoch + 1
                best_state_dict = copy.deepcopy(model.state_dict())
                log_str += "  | best*"

        print(log_str)

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    history['best_epoch'] = best_epoch
    history['best_metric'] = best_metric
    return model, history


print("训练函数定义完成")

训练函数定义完成


## 6. 构建模型并训练

In [20]:
rankmixer_config = {
    'emb_dim': 8,
    'content_emb_dim': CONTENT_EMB_DIM,
    'max_hist_len': MAX_HIST_LEN,
    'num_T': 4,
    'num_D': 64,
    'num_H': 4,
    'num_layers': 2,
    'expansion_ratio': 4,
    'logit_head_units': [32],
    'dropout_rate': 0.1,
    'alpha': 0.5,
    'pos_weight': 1.0,
    'neg_weight': 1.0,
    'use_linear': True,
}

model = RankMixerRanking(sparse_vocab_sizes, dense_input_dim, rankmixer_config).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\n设备: {DEVICE}")
print(f"可训练参数量: {total_params:,}")
print(f"RankMixer 输入维度: {model.rankmixer_input_dim}")

RankMixerRanking(
  (sparse_embeddings): ModuleDict(
    (user_id): Embedding(149760, 8)
    (click_article_id): Embedding(22783, 8, padding_idx=0)
    (category_id): Embedding(248, 8)
    (click_environment): Embedding(3, 8)
    (click_deviceGroup): Embedding(4, 8)
    (click_os): Embedding(8, 8)
    (click_country): Embedding(11, 8)
    (click_region): Embedding(28, 8)
    (click_referrer_type): Embedding(7, 8)
  )
  (linear_sparse_embeddings): ModuleDict(
    (user_id): Embedding(149760, 1)
    (click_article_id): Embedding(22783, 1, padding_idx=0)
    (category_id): Embedding(248, 1)
    (click_environment): Embedding(3, 1)
    (click_deviceGroup): Embedding(4, 1)
    (click_os): Embedding(8, 1)
    (click_country): Embedding(11, 1)
    (click_region): Embedding(28, 1)
    (click_referrer_type): Embedding(7, 1)
  )
  (linear_dense): Linear(in_features=68, out_features=1, bias=True)
  (content_emb_table): Embedding(364048, 64, padding_idx=0)
  (position_emb): Embedding(50, 64)
  (ra

In [21]:
print("开始训练 RankMixer 模型...")
print(f"训练集样本数: {len(y_trn)}")
if offline:
    print(f"验证集样本数: {len(y_val)}")
print()

model, history = train_rankmixer_model(
    model, x_trn, y_trn,
    x_val=x_val if offline else None,
    y_val=y_val if offline else None,
    val_df=val_df if offline else None,
    val_df_hit=val_df_hit if offline else None,
    val_hit_mask=val_hit_mask if offline else None,
    epochs=5,
    batch_size=256,
    lr=0.001,
    alpha=rankmixer_config['alpha'],
    pos_weight=rankmixer_config['pos_weight'],
    neg_weight=rankmixer_config['neg_weight'],
    select_topk=5,
)

if history.get('best_epoch') is not None:
    print(f"\nBest checkpoint epoch: {history['best_epoch']}")
    print(f"Best hit MRR@5: {history['best_metric']:.5f}")

print("\nRankMixer 模型训练完成！")

开始训练 RankMixer 模型...
训练集样本数: 1426880
验证集样本数: 8000000

Epoch 1/5 (118.0s)  Total: 2.4816  CE: 0.3568  GE: 4.6064  | Full MRR@5: 0.13558  | Hit MRR@5: 0.19806  | Full NDCG@5: 0.1659  | Hit NDCG@5: 0.2424  | best*
Epoch 2/5 (117.5s)  Total: 2.2824  CE: 0.3013  GE: 4.2634  | Full MRR@5: 0.16844  | Hit MRR@5: 0.24605  | Full NDCG@5: 0.2019  | Hit NDCG@5: 0.2950  | best*
Epoch 3/5 (118.8s)  Total: 2.2220  CE: 0.2856  GE: 4.1584  | Full MRR@5: 0.17088  | Hit MRR@5: 0.24963  | Full NDCG@5: 0.2034  | Hit NDCG@5: 0.2971  | best*
Epoch 4/5 (116.1s)  Total: 2.1828  CE: 0.2748  GE: 4.0907  | Full MRR@5: 0.16942  | Hit MRR@5: 0.24749  | Full NDCG@5: 0.2024  | Hit NDCG@5: 0.2956
Epoch 5/5 (116.8s)  Total: 2.1488  CE: 0.2649  GE: 4.0328  | Full MRR@5: 0.18349  | Hit MRR@5: 0.26804  | Full NDCG@5: 0.2174  | Hit NDCG@5: 0.3176  | best*

Best checkpoint epoch: 5
Best hit MRR@5: 0.26804

RankMixer 模型训练完成！


## 7. 最终评估与对比

In [22]:
if offline and x_val is not None:
    print("=" * 70)
    print("RankMixer 验证集最终评估")
    print("=" * 70)

    for mode in ['fusion', 'calibration', 'ranking']:
        metrics = evaluate_validation(
            model, x_val, y_val, val_df,
            val_df_hit=val_df_hit, val_hit_mask=val_hit_mask,
            batch_size=256, topk=5, mode=mode,
        )
        print(f"\n[{mode}]")
        print(f"  Full AUC: {metrics.get('full_auc', 0):.5f}")
        print(f"  Full MRR@5: {metrics.get('full_mrr5', 0):.5f}")
        print(f"  Full NDCG@5: {metrics.get('full_ndcg5', 0):.4f}")
        print(f"  Full HR@5: {metrics.get('full_hr5', 0):.5f}")
        print(f"  Hit MRR@5: {metrics.get('hit_mrr5', 0):.5f}")
        print(f"  Hit NDCG@5: {metrics.get('hit_ndcg5', 0):.4f}")
        print(f"  Hit HR@5: {metrics.get('hit_hr5', 0):.5f}")

    print("\n" + "=" * 70)
    print("JRC baseline 参考 (从 jrc-ranking_all.ipynb best epoch 5):")
    print("  Hit MRR@5: 0.27117")
    print("  Full MRR@5: 0.18563")
    print("  Hit NDCG@5: 0.3202")
    print("  Hit HR@5: 0.46965")
    print("=" * 70)

RankMixer 验证集最终评估

[fusion]
  Full AUC: 0.83176
  Full MRR@5: 0.18349
  Full NDCG@5: 0.2174
  Full HR@5: 0.32060
  Hit MRR@5: 0.26804
  Hit NDCG@5: 0.3176
  Hit HR@5: 0.46834

[calibration]
  Full AUC: 0.83176
  Full MRR@5: 0.18349
  Full NDCG@5: 0.2174
  Full HR@5: 0.32060
  Hit MRR@5: 0.26804
  Hit NDCG@5: 0.3176
  Hit HR@5: 0.46834

[ranking]
  Full AUC: 0.83176
  Full MRR@5: 0.18349
  Full NDCG@5: 0.2174
  Full HR@5: 0.32060
  Hit MRR@5: 0.26805
  Hit NDCG@5: 0.3176
  Hit HR@5: 0.46834

JRC baseline 参考 (从 jrc-ranking_all.ipynb best epoch 5):
  Hit MRR@5: 0.27117
  Full MRR@5: 0.18563
  Hit NDCG@5: 0.3202
  Hit HR@5: 0.46965
